# Étape 4 : Interprétabilité — Analyse SHAP

## SHAP (SHapley Additive exPlanations)

**Fondement théorique** : Valeurs de Shapley issues de la théorie des jeux coopératifs.

$$\phi_i(f) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ f(S \cup \{i\}) - f(S) \right]$$

**Interprétation** : $\phi_i$ = contribution marginale moyenne de la feature $i$ sur toutes les coalitions possibles.

### Propriétés clés
| Propriété | Signification |
|-----------|---------------|
| **Efficiency** | $\sum_i \phi_i = f(x) - E[f(x)]$ (les SHAP somment à la prédiction) |
| **Symmetry** | Features identiques → mêmes SHAP |
| **Dummy** | Feature sans impact → SHAP = 0 |
| **Additivity** | Cohérence entre modèles combinés |

### Deux explainers utilisés
- **TreeSHAP** (XGBoost) : exact, $O(TLD^2)$, très rapide
- **LinearSHAP** (Régression Logistique) : exact pour les modèles linéaires

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import joblib
from IPython.display import display, Image

from config import *
from eda import run_eda
from interpretability import run_interpretability, get_feature_names

# Rechargement
eda_data = run_eda(OUTPUT_DIR)
X_test  = eda_data['X_test']
y_test  = eda_data['y_test']
feature_cols = eda_data['feature_cols']

model_lr  = joblib.load(os.path.join(OUTPUT_DIR, 'model_logistic.pkl'))
model_xgb = joblib.load(os.path.join(OUTPUT_DIR, 'model_xgboost.pkl'))

print(f'Test set : {X_test.shape[0]:,} transactions, {y_test.sum()} fraudes')
print(f'Features : {len(feature_cols)}')

## 4.1 TreeSHAP — XGBoost

**TreeSHAP** exploite la structure arborescente pour calculer les valeurs exactes de Shapley en temps polynomial (vs exponentiel pour KernelSHAP).

Pour chaque prédiction $f(x)$ :
$$f(x) = E[f(X)] + \sum_{i=1}^{d} \phi_i$$

Où $E[f(X)] \approx$ fréquence de fraude dans les données d'entraînement.

In [ ]:
from interpretability import compute_shap_xgboost

feature_names = list(feature_cols)
shap_vals_xgb, mean_shap_xgb = compute_shap_xgboost(
    model_xgb, X_test, y_test, feature_names, OUTPUT_DIR
)

print(f'SHAP values shape : {shap_vals_xgb.shape}')
print(f'Valeur absolue moyenne par feature (top 5) :')
top5_idx = np.argsort(mean_shap_xgb)[-5:][::-1]
for i in top5_idx:
    fname = feature_names[i] if i < len(feature_names) else f'f{i}'
    print(f'  {fname:<20}: {mean_shap_xgb[i]:.4f}')

In [ ]:
print('Beeswarm Summary Plot — Impact de chaque feature sur la prédiction :')
print('  Rouge = valeur élevée de la feature → prédit fraude')
print('  Bleu  = valeur faible  de la feature → prédit normal')
display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_summary_beeswarm.png')))

In [ ]:
print('Bar Plot — Importance globale (|SHAP| moyen) :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_bar_importance.png')))

In [ ]:
print('Dependence Plots — Relation feature/SHAP pour les 3 features les plus importantes :')
print('  Couleur = classe réelle (Rouge=Fraude, Bleu=Normal)')
display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_dependence_top3.png')))

## 4.2 Waterfall Plot — Explication d'une fraude individuelle

Le waterfall plot explique **pourquoi** le modèle a prédit "fraude" pour une transaction spécifique.

- **Rouge** : feature qui pousse la prédiction vers Fraude
- **Bleu** : feature qui pousse la prédiction vers Normal
- La somme des SHAP = prédiction - baseline

In [ ]:
if os.path.exists(os.path.join(OUTPUT_DIR, 'shap_waterfall_fraud.png')):
    print('Waterfall Plot — Transaction frauduleuse individuelle :')
    display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_waterfall_fraud.png')))
else:
    print('Waterfall plot non disponible (aucune fraude dans le sous-échantillon).')

## 4.3 Comparaison SHAP par classe

Ceci révèle les **différences structurelles** entre les patterns frauduleux et normaux :
- Features avec SHAP moyen positif pour Fraude → indicateurs de fraude
- Features avec SHAP moyen négatif pour Fraude → indicateurs de normalité

In [ ]:
print('Comparaison SHAP moyen par classe :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_class_comparison.png')))

## 4.4 LinearSHAP — Régression Logistique

Pour les modèles linéaires :
$$\phi_i = w_i \cdot (x_i - E[x_i])$$

Les valeurs SHAP sont directement proportionnelles aux coefficients × (feature - moyenne).
Cela permet de valider que les coefficients ElasticNet capturent les bons signaux.

In [ ]:
from interpretability import compute_shap_logistic

shap_vals_lr, mean_shap_lr = compute_shap_logistic(
    model_lr, X_test, y_test, feature_names, OUTPUT_DIR
)

print('SHAP Logistic Regression :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'shap_logistic_importance.png')))

## 4.5 Cohérence entre SHAP et coefficients logistiques

In [ ]:
# Top features selon SHAP XGBoost vs Logistic
print('=== TOP 10 FEATURES PAR MODÈLE ===')
feature_names_clean = get_feature_names(X_test.shape[1])

top10_xgb = np.argsort(mean_shap_xgb)[-10:][::-1]
top10_lr  = np.argsort(mean_shap_lr)[-10:][::-1]

df_compare = pd.DataFrame({
    'Rank': range(1, 11),
    'XGBoost (TreeSHAP)': [feature_names_clean[i] if i < len(feature_names_clean) else f'f{i}' for i in top10_xgb],
    'Logistic (LinearSHAP)': [feature_names_clean[i] if i < len(feature_names_clean) else f'f{i}' for i in top10_lr],
})
display(df_compare)

# Intersection
xgb_set = set(top10_xgb[:5])
lr_set  = set(top10_lr[:5])
common = xgb_set & lr_set
print(f'\nFeatures communes dans le top 5 : {len(common)}/5')
if common:
    for i in common:
        fname = feature_names_clean[i] if i < len(feature_names_clean) else f'f{i}'
        print(f'  → {fname}')

## Résumé de l'étape 4

### Conclusions SHAP

1. **V14** est systématiquement le principal indicateur de fraude — sa valeur négative est fortement associée aux fraudes (cohérent avec l'analyse de corrélation de l'EDA)

2. **V4** et **V12** sont également très discriminants — des valeurs élevées/faibles créent un signal fort

3. **Amount_log** contribue peu globalement mais peut être décisif sur des cas limites (montants atypiques)

4. **Cohérence** entre les deux modèles : les features importantes pour XGBoost (TreeSHAP) le sont aussi pour la Régression Logistique (LinearSHAP), ce qui valide la robustesse des signaux

### Avantages SHAP vs méthodes alternatives
| Méthode | Globale | Locale | Exacte | Rapide |
|---------|---------|--------|--------|--------|
| **TreeSHAP** | ✓ | ✓ | ✓ | ✓ |
| LIME | ✗ | ✓ | ✗ (approx) | ✓ |
| Permutation | ✓ | ✗ | ✗ | ✗ |
| Coefficients LR | ✓ | Partiel | ✓ | ✓ |